In [ ]:
import os
from dotenv import load_dotenv
from uuid import uuid4

from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

from langchain_google_genai import ChatGoogleGenerativeAI


In [ ]:

load_dotenv()


In [ ]:
groq_llm = ChatGroq(
    model="llama-3.1-8b-instant",  # also: mixtral-8x7b-32768, gemma2-9b-it
    temperature=0,
    api_key=os.getenv("GROQ_API"),
)

response = groq_llm.invoke([HumanMessage(content="What is LangChain used for? Answer in 2 sentences.")])
print("Groq:", response.content)

In [ ]:

gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=os.getenv("GOOG_API"),
)

response = gemini_llm.invoke("What is LangChain used for? Answer in 2 sentences.")
print("Gemini:", response.content)

---
## RAG Pipeline
Loads all `.md` files from `/memory`, chunks them, embeds with a local HuggingFace model, stores in FAISS, then answers questions using Groq.

In [ ]:
!pip install sentence-transformers faiss-cpu -q

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader

load_dotenv()

MEMORY_DIR = Path("memory")

loader = DirectoryLoader(
    str(MEMORY_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8", "autodetect_encoding": True},
    show_progress=True,
    use_multithreading=True,
)
docs = loader.load()
print(f"Loaded {len(docs)} documents")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
    separators=[", ", " "],
)
chunks = splitter.split_documents(docs)

print(f"Split into {len(chunks)} chunks")
print(f"Sample chunk from '{Path(chunks[0].metadata['source']).name}':")
print(chunks[0].page_content[:300])

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Runs locally — no API key needed
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("Building FAISS index...")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("memory_index")
print(f"Done — indexed {vectorstore.index.ntotal} vectors, saved to ./memory_index/")

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
    api_key=os.getenv("GROQ_API"),
)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a personal assistant with access to the user's private notes and journal entries.
Answer the question using only the provided context. Be specific and reference the source documents where relevant.
If the context doesn't contain enough information, say so.

Context:
{context}"""),
    ("human", "{question}"),
])

def format_docs(docs):
    return "".join(f"[{Path(d.metadata['source']).name}]{d.page_content}" for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG pipeline ready.")

In [ ]:
# --- Run a query ---
question = "describe my personality based on all the documents you've read about me"

answer = rag_chain.invoke(question)
print(f"Q: {question}")
print(f"A: {answer}")

In [ ]:
# --- Optional: reload index from disk instead of rebuilding ---
# vectorstore = FAISS.load_local(
#     "memory_index",
#     embeddings,
#     allow_dangerous_deserialization=True,
# )

## Wiki Pipeline

In [ ]:
from pydantic import BaseModel
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
class Entity(BaseModel):
    name: str
    type: Literal["person", "concept", "event", "project"]
    slug: str
    relevance: str # one sentence: how this entity appears in the document

class ExtractionResult(BaseModel):
    entities: list[Entity]

In [ ]:
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """Extract every notable entity from this document that warrants its own wiki page.
Only include entities that are substantive — not passing mentions.
For each entity, assign a type: person, concept, event, or project.
The slug must be kebab-case (e.g. 'john-douglas', 'procrastination')."""),
    ("human", "{document}"),
])


structured_llm = gemini_llm.with_structured_output(ExtractionResult)
extraction_chain = extraction_prompt | structured_llm 


In [ ]:
raw_text = "something is going to happen to Jason Dunby today that I can forsee. It doesn't look beautiful to me. Phantasms of strange aparitions keep appearing in sight every now and then. This is all going to quickly for me to comprehend."
result = extraction_chain.invoke({"document": raw_text})

In [ ]:
print(result)

In [ ]:
from pathlib import Path

TYPE_DIRS = {"person": "people", "concept": "concepts", "event": "events", "project": "projects"}
WIKI_DIR = Path("./wiki")


def wiki_path(entity: Entity) -> Path:
    return WIKI_DIR / TYPE_DIRS[entity.type] / f"{entity.slug}.md"


In [ ]:
import re

def parse_description(page_path: Path) -> tuple[str, str]:
    """Extract title and description from a wiki page's first heading and paragraph."""

    text = page_path.read_text(encoding="utf-8")
    title_match = re.search(r'^#\s+(.+)$', text, re.MULTILINE)
    desc_match = re.search(r'^(?!#)(.{20,})', text, re.MULTILINE)
    title = title_match.group(1) if title_match else page_path.stem
    desc = desc_match.group(1)[:80] if desc_match else ""
    return title, desc


def rebuild_index(wiki_dir: Path) -> None:
    sections = {"people": "People", "concepts": "Concepts", "events": "Events", "projects": "Projects"}
    lines = ["# Wiki Index\n"]

    for folder, heading in sections.items():
        folder_path = wiki_dir / folder
        if not folder_path.exists():
            continue
        pages = sorted(folder_path.glob("*.md"))
        if not pages:
            continue
        lines.append(f"## {heading}\n")
        for page in pages:
            title, desc = parse_description(page)
            rel_path = f"{folder}/{page.name}"
            lines.append(f"- [{title}]({rel_path}) — {desc}")
        lines.append("")

    (wiki_dir / "index.md").write_text("\n".join(lines), encoding="utf-8")
